In [ ]:
from Bio import Entrez
import time
import os
from http.client import IncompleteRead
import glob
import json
import re
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from vllm import LLM, SamplingParams

In [ ]:
Entrez.email = "your_email@example.com"
save_dir = "./data/pubmed"

query = "catalase"

start_year = 2025
end_year = 2001

retmax_per_year = 10000
batch_size = 1000

In [ ]:
os.makedirs(save_dir, exist_ok=True)

In [ ]:
def fetch_with_retry(db, ids, rettype, retmode, retries=3):
    for attempt in range(retries):
        try:
            handle = Entrez.efetch(db=db, id=ids, rettype=rettype, retmode=retmode)
            data = handle.read()
            handle.close()
            return data
        except IncompleteRead:
            print(f"IncompleteRead error, retrying attempt {attempt+1}...")
            time.sleep(2)
    return ""

In [ ]:
for year in range(start_year, end_year - 1, -1):
    print(f"\nFetching PMIDs for year {year}...")

    year_query = f"{query} AND {year}[PDAT]"
    handle = Entrez.esearch(db="pubmed", term=year_query, retmax=retmax_per_year)
    record = Entrez.read(handle)
    pmid_list = record["IdList"]
    count = int(record["Count"])

    print(f"  Total publications this year: {count}, PMIDs retrieved: {len(pmid_list)}")

    if not pmid_list:
        continue

    out_file = os.path.join(save_dir, f"medline_{year}.txt")
    with open(out_file, "w", encoding="utf-8") as out_f:
        for start in range(0, len(pmid_list), batch_size):
            end = min(start + batch_size, len(pmid_list))
            batch_pmids = pmid_list[start:end]
            ids = ",".join(batch_pmids)

            print(f"  Fetching MEDLINE records for year {year}, entries {start+1} - {end}...")

            data = fetch_with_retry("pubmed", ids, "medline", "text")
            if not data:
                print(f"Failed to fetch entries {start+1}-{end}, skipping...")
                continue

            out_f.write(f"=== {year} MEDLINE entries {start+1} - {end} ===\n")
            out_f.write(data)
            out_f.write("\n" + "="*80 + "\n")

            time.sleep(0.4)

print("\nCompleted fetching MEDLINE abstracts for all years. Each year saved as a separate file.")

In [ ]:
def extract_year(filename):
    match = re.search(r"medline_(\d{4})\.txt", filename)
    if match:
        return int(match.group(1))
    else:
        return 0

def parse_medline_blocks(blocks):
    records = []
    for block in blocks:
        pmid = None
        dp = None
        title_lines = []
        abstract_lines = []
        doi = None
        current_field = None

        for l in block.split("\n"):
            if l.startswith("PMID-"):
                pmid = l.replace("PMID-", "").strip()
                current_field = None
            elif l.startswith("DP  -"):
                dp_line = l.replace("DP  -", "").strip()
                year_match = re.search(r"\b(\d{4})\b", dp_line)
                dp = year_match.group(1) if year_match else None
                current_field = None
            elif l.startswith("TI  - "):
                title_lines.append(l.replace("TI  - ", "").strip())
                current_field = "TI"
            elif l.startswith("AB  - "):
                abstract_lines.append(l.replace("AB  - ", "").strip())
                current_field = "AB"
            elif l.startswith("LID - ") and "[doi]" in l:
                doi_match = re.search(r"(10\.\d{4,9}/\S+)", l)
                if doi_match:
                    doi = doi_match.group(1)
                current_field = None
            elif current_field == "TI" and l.startswith("      "):
                title_lines.append(l.strip())
            elif current_field == "AB" and l.startswith("      "):
                abstract_lines.append(l.strip())
            else:
                current_field = None

        records.append({
            "PMID": pmid,
            "DP": dp,
            "Title": " ".join(title_lines).strip(),
            "Abstract": " ".join(abstract_lines).strip(),
            "DOI": doi
        })
    return records

def read_blocks_from_file(filepath):
    blocks = []
    current_block = []
    inside_block = False

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            stripped = line.strip()
            if stripped.startswith("PMID-"):
                inside_block = True
                current_block = [line.rstrip("\n")]
            elif inside_block:
                if stripped == "":
                    blocks.append("\n".join(current_block))
                    inside_block = False
                else:
                    current_block.append(line.rstrip("\n"))
        if inside_block and current_block:
            blocks.append("\n".join(current_block))
    return blocks

In [ ]:
all_records = []

file_list = glob.glob("./data/pubmed/medline_*.txt")
file_list.sort(key=extract_year)

year_totals = defaultdict(int)
year_with_ab = defaultdict(int)
year_without_ab = defaultdict(int)

for filepath in file_list:
    blocks = read_blocks_from_file(filepath)

    ab_blocks = [blk for blk in blocks
                 if any(line.strip().startswith("AB  -") for line in blk.splitlines())]
    records = parse_medline_blocks(ab_blocks)
    all_records.extend(records)

    for blk in blocks:
        dp_year = None
        for line in blk.splitlines():
            if line.startswith("DP  -"):
                dp_line = line.replace("DP  -", "").strip()
                m = re.search(r"\b(\d{4})\b", dp_line)
                dp_year = int(m.group(1)) if m else None
                break

        if dp_year is None:
            continue

        has_ab = any(line.strip().startswith("AB  -") for line in blk.splitlines())
        year_totals[dp_year] += 1
        if has_ab:
            year_with_ab[dp_year] += 1
        else:
            year_without_ab[dp_year] += 1

years_sorted = sorted(year_totals.keys())
yearly_stats = [{
    "year": y,
    "total_blocks": year_totals[y],
    "with_abstract": year_with_ab[y],
    "without_abstract": year_without_ab[y],
} for y in years_sorted]

In [ ]:
df_yearly = pd.DataFrame(yearly_stats).sort_values("year").reset_index(drop=True)

In [ ]:
with open("./data/pubmed.json", "w", encoding="utf-8") as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)
print(f"{len(all_records)} records saved to ./data/pubmed.json")

In [ ]:
mp = pd.read_excel("./data/materials_project.xlsx")

In [ ]:
from pymatgen.core import Composition
from pymatgen.core.periodic_table import Element

def check_composition(formula):
    try:
        comp = Composition(formula)
        elements = comp.elements
        
        contains_tm = any(Element(el.symbol).is_transition_metal for el in elements)
        is_elemental = len(elements) == 1
        
        return pd.Series({
            'contains_transition_metal': contains_tm,
            'is_elemental': is_elemental
        })
    except:
        return pd.Series({
            'contains_transition_metal': False,
            'is_elemental': False
        })

In [ ]:
mp[['contains_transition_metal', 'is_elemental']] = mp['formula'].apply(check_composition)

In [ ]:
filtered_df = mp[(mp['contains_transition_metal']) & (~mp['is_elemental'])]

In [ ]:
pubmed = json.load(open('./data/pubmed.json', 'r'))

In [ ]:
def add_parentheses_to_digits(formula):
    def repl(match):
        num = match.group()
        if num == '1':
            return num
        else:
            return f'({num})'

    return re.sub(r'\d+', repl, formula)

In [ ]:
def formula_in_text_no_alpha_around(formula, text):
    pattern = rf'(?<![A-Za-z0-9\)]){re.escape(formula)}(?![A-Za-z0-9\(])'
    return re.search(pattern, text) is not None

In [ ]:
formula_pmids = {}
substances = filtered_df.formula.tolist()

for i, formula in enumerate(tqdm(substances, desc="Processing formulas"), 1):
    formula_variants = [formula, add_parentheses_to_digits(formula)]

    matched_pmids = []
    for item in pubmed:
        abstract_text = item.get('Abstract', '')
        if any(fv in abstract_text for fv in formula_variants):
            matched_pmids.append(item['PMID'])

    formula_pmids[formula] = matched_pmids

In [ ]:
filtered_pmids = []
for formula, pmid_list in formula_pmids.items():
    if pmid_list:
        for pmid in pmid_list:
            filtered_pmids.append({'formula': formula, 'pmid': pmid})

In [ ]:
pmid_to_abstract = {item['PMID']: item.get('Abstract', '') for item in pubmed}

In [ ]:
def formula_in_text_no_alpha_around(formula, text):
    pattern = rf'(?<![A-Za-z0-9\)]){re.escape(formula)}(?![A-Za-z0-9\(])'
    return re.search(pattern, text) is not None

for entry in tqdm(filtered_pmids, desc="Checking formulas in abstracts"):
    formula = entry['formula']
    pmid = entry['pmid']
    
    abstract_text = pmid_to_abstract.get(pmid, '')
    
    entry['abstract'] = abstract_text
    
    formula_variants = [formula, add_parentheses_to_digits(formula)]

    entry['formula_in_abstract'] = any(formula_in_text_no_alpha_around(fv, abstract_text) for fv in formula_variants)

In [ ]:
filtered_pmids_v2 = [{'formula':item['formula'], 'pmids':item['pmid'], 'abstract':item['abstract']} for item in filtered_pmids if item['formula_in_abstract'] == True]

In [ ]:
save_path = "./data/pubmed_v2.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(filtered_pmids_v2, f, ensure_ascii=False, indent=2)

In [ ]:
model_path = "/ssd/common/LLMs/Qwen2.5-32B-Instruct"
llm = LLM(model=model_path, tensor_parallel_size=4)

In [ ]:
sampling_params = SamplingParams(max_tokens=2048, temperature=0.7, top_p=0.9, repetition_penalty=1.1)

In [ ]:
FEW_SHOT_PROMPT = """You are an expert in materials science and enzyme catalysis.
You are helping to build a database of materials with catalase-like (CAT) activity.

For each paper, you will be given the formula of a material and corresponding abstract of a scientific paper.
Your task is to carefully analyze the abstract and determine whether it states that the given material has catalase-like (CAT) activity.

At the end of your response, please clearly state the final answer in the following format exactly:
If you can determine definitively, answer with either:
The final answer is **[YES]**.
or
The final answer is **[NO]**.
If you cannot determine definitively, answer with:
The final answer is **[Uncertain]**.

Now classify the following abstract:
Abstract: {abstract}
Formula: {formula}
Answer:
"""

In [ ]:
batch_prompts = []
for item in filtered_pmids_v2:
    batch_prompts.append(FEW_SHOT_PROMPT.format(abstract=item['abstract'], formula=item['formula']))

In [ ]:
outputs = llm.generate(batch_prompts, sampling_params)

In [ ]:
texts = [output.outputs[0].text.strip() for output in outputs]

In [ ]:
for i in range(len(filtered_pmids_v2)):
    filtered_pmids_v2[i]['answer'] = texts[i]

In [ ]:
filtered_pmids_v2 = json.load(open('./data/pubmed_v3.json', 'r'))

In [ ]:
for item in filtered_pmids_v2:
    answer = item.get('answer', '')
    answer_lower = answer.lower()

    states = []
    if '**[yes]**' in answer_lower:
        states.append('yes')
    if '**[no]**' in answer_lower:
        states.append('no')
    if '**[uncertain]**' in answer_lower:
        states.append('uncertain')

    if not states:
        states = ['other']
    
    item['states'] = states

In [ ]:
filtered_pmids_v3 = [entry for entry in filtered_pmids_v2 if entry.get("states") == ["yes"]]

In [ ]:
save_path = "./data/pubmed_v3.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(filtered_pmids_v3, f, ensure_ascii=False, indent=2)

In [ ]:
pmids = json.load(open('./data/pubmed.json', 'r'))
pmids_v2 = json.load(open('./data/pubmed_v2.json', 'r'))
pmids_v3 = json.load(open('./data/pubmed_v3.json', 'r'))

In [ ]:
pmid_to_dp = {str(item["PMID"]): item.get("DP") for item in pmids}

for entry in pmids_v2:
    m = re.search(r"\d+", str(entry.get("pmids", "")))
    entry["DP"] = pmid_to_dp.get(m.group(0)) if m else None

In [ ]:
for entry in pmids_v3:
    m = re.search(r"\d+", str(entry.get("pmids", "")))
    entry["DP"] = pmid_to_dp.get(m.group(0)) if m else None

In [ ]:
dp_counts = pd.to_numeric(pd.DataFrame(pmids)["DP"], errors="coerce")\
                .loc[lambda s: s.between(2001, 2025)]\
                .value_counts().sort_index()

In [ ]:
dp_counts_v2 = pd.to_numeric(pd.DataFrame(pmids_v2)["DP"], errors="coerce")\
                .loc[lambda s: s.between(2001, 2025)]\
                .value_counts().sort_index()

In [ ]:
dp_counts_v3 = pd.to_numeric(pd.DataFrame(pmids_v3)["DP"], errors="coerce")\
                .loc[lambda s: s.between(2001, 2025)]\
                .value_counts().sort_index()

In [ ]:
dp_all = pd.concat(
    [
        dp_counts.rename("with abstract"),
        dp_counts_v2.rename("with formula"),
        dp_counts_v3.rename("with nanozyme")
    ],
    axis=1
).sort_index()

In [ ]:
df_yearly = pd.DataFrame(yearly_stats)[["year", "total_blocks"]].set_index("year")

In [ ]:
df_yearly.index = pd.to_numeric(df_yearly.index, errors="coerce")
dp_all.index = pd.to_numeric(dp_all.index, errors="coerce")

In [ ]:
dp_all.insert(0, "total_blocks", df_yearly["total_blocks"].reindex(dp_all.index))

In [ ]:
dp_diff = dp_all.iloc[:, :-1].sub(dp_all.iloc[:, 1:].to_numpy(), axis=0)
dp_diff.columns = [f"{c1}_minus_{c2}" for c1, c2 in zip(dp_all.columns[:-1], dp_all.columns[1:])]
dp_diff["with nanozyme"] = dp_all["with nanozyme"]

In [ ]:
dp_diff.columns = [
    "without abstract",
    "without formula",
    "without nanozyme",
    "final selection"
]

In [ ]:
dp_diff